# 03 Label construction

**Purpose:** build the targets the models will learn from. Neither Task 1 target is given directly, so we derive them from the route legs. Task 2A also needs a weekly demand table built to Waypoint's counting rules.

**Inputs:** clean tables from notebook 02 (`data/processed/`).

**Outputs** (in `data/processed/`):
- `stops.parquet`: one row per dispatched order (train and test) joined to its route leg, with the two Task 1 labels filled in for train
- `weekly_demand.parquet`: weekly total and chilled volume per depot and brand, ready for Task 2A

**The labels in one line each**
- `service_min` = time the vehicle leaves the outlet minus the later of its arrival and the window opening
- `late` = 1 if the vehicle arrives after the window closes, else 0

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from xora.io import load_processed, save_processed
from xora.plotting import apply_style, SERIES
from xora import checks as ck

apply_style()
pd.set_option("display.width", 160)

orders = load_processed("orders")
legs = load_processed("legs")
outlets = load_processed("outlets")
calendar = load_processed("calendar")
allowance = load_processed("service_allowance")

## 1. One row per delivered order

Notebook 01 showed that every dispatched order, deferred ones included, has exactly one route leg. We join on route and stop position, and keep only the leg columns the labels and later features need.

In [2]:
LEG_COLS = [
    "route_id", "seq", "leg_id", "date", "from_point", "distance_km", "dow", "monsoon",
    "planned_depart_time_min", "planned_travel_duration_min", "planned_arrival_time_min",
    "actual_depart_time_min", "actual_travel_duration_min", "arrival_time_min", "leave_outlet_time_min",
]

dispatched = orders[orders.route_id.notna()]
stops = dispatched.merge(
    legs[LEG_COLS], left_on=["route_id", "seq_in_route"], right_on=["route_id", "seq"],
    how="left", validate="one_to_one",
).merge(outlets[["outlet_id", "dock_type", "parking_constraint"]], on="outlet_id", how="left", validate="many_to_one")

stops.groupby("split").size().rename("stops").to_frame()

,stops
split,
test,5014
train,91894


`validate="one_to_one"` makes pandas raise an error if any order matched more than one leg, so the join is checked by construction. The `not_run` orders drop out here because they never had a route. They come back in section 5 for the demand table.